# ÉTAPE 1 — Analyse Exploratoire des Données (EDA) Complète

Ce notebook présente l'analyse exploratoire complète du dataset `tunisie_meteo_reelle_2009_2026.csv`. L'objectif est d'explorer les caractéristiques météorologiques de 4 villes tunisiennes (Tunis, Sfax, Sousse, Bizerte) pour comprendre les distributions, la saisonnalité, les corrélations et le déséquilibre de classe sur la variable cible `pluie_demain_bin`.

## Plan de l'analyse :
1. **Configuration et Chargement** des données.
2. **Analyse du Déséquilibre de Classe** de la cible `pluie_demain_bin`.
3. **Statistiques Descriptives** groupées par ville.
4. **Analyse des Valeurs Manquantes** et stratégie de traitement.
5. **Distributions des Variables Clés** (`temp_mean`, `precipitation`, `humidite_mean`) par ville et par saison.
6. **Matrice de Corrélation** (heatmap des features numériques).
7. **Visualisation de la Saisonnalité** (boxplots mensuels par ville).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style premium pour les graphiques
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 15,
    "figure.dpi": 120
})

# Palettes de couleurs
SEASONS_PALETTE = {
    "Hiver": "#1f77b4",
    "Printemps": "#2ca02c",
    "Été": "#ff7f0e",
    "Automne": "#9467bd"
}
CLASS_PALETTE = ["#8da0cb", "#fc8d62"]

## 1. Chargement des Données

In [ ]:
# Chemin d'accès au dataset
data_path = "../data/tunisie_meteo_reelle_2009_2026.csv"
df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"])

print(f"Dimensions du dataset : {df.shape[0]:,} lignes, {df.shape[1]} colonnes.")
df.head()

## 2. Analyse du Déséquilibre de Classe (`pluie_demain_bin`)

Il est crucial de quantifier la proportion de jours avec pluie demain par rapport aux jours secs pour adapter notre stratégie de modélisation (ex. ajustement des poids, métriques d'évaluation adaptées comme le F1-score ou l'Average Precision).

In [ ]:
target_col = "pluie_demain_bin"
vc = df[target_col].value_counts(dropna=False)
vc_pct = df[target_col].value_counts(normalize=True, dropna=False) * 100

print("Distribution de la cible :
")
for idx, val in vc.items():
    print(f"  Classe {idx}: {val:5,d} ({vc_pct[idx]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(data=df, x=target_col, palette=CLASS_PALETTE, ax=ax)
ax.set_title("Distribution de la cible 'pluie_demain_bin'")
ax.set_xlabel("Il pleuvra demain (0 = Non, 1 = Oui)")
ax.set_ylabel("Nombre de jours")

# Ajouter les valeurs au-dessus des barres
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f"{height:,.0f}\n({height/len(df)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2., height + 100),
                ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 3. Statistiques Descriptives par Ville

In [ ]:
desc_cols = ["temp_max", "temp_min", "temp_mean", "precipitation", "humidite_mean", "vent_max", "pression_mean"]

# Moyennes globales par ville
print("Moyennes des variables météorologiques par ville :
")
print(df.groupby("ville")[desc_cols].mean().round(2))

# Description complète par ville
df.groupby("ville")[desc_cols].describe().transpose().round(2)

## 4. Analyse des Valeurs Manquantes

Analysons le nombre de valeurs manquantes par colonne pour concevoir notre stratégie d'imputation dans l'étape suivante.

In [ ]:
na_counts = df.isnull().sum()
na_pcts = (df.isnull().sum() / len(df)) * 100
na_df = pd.DataFrame({"Missing Count": na_counts, "Percentage (%)": na_pcts})
na_df = na_df.sort_values(by="Missing Count", ascending=False)

print("Top 15 des colonnes avec le plus de valeurs manquantes :
")
print(na_df.head(15))

print(f"
Nombre total de lignes avec au moins une valeur manquante : {df.isnull().any(axis=1).sum():,}")

> **Stratégie d'imputation proposée (Étape 2) :** 
> Comme les variables météo dépendent fortement du climat local et de la période de l'année, nous imputerons les valeurs manquantes par la **médiane spécifique à la ville et au mois** (ex. la médiane de Tunis en Décembre pour les données manquantes de Tunis en Décembre).

## 5. Distributions de `temp_mean`, `precipitation`, et `humidite_mean` par Ville et Saison

In [ ]:
features_to_plot = ["temp_mean", "precipitation", "humidite_mean"]
villes = df["ville"].unique()

fig, axes = plt.subplots(nrows=len(villes), ncols=len(features_to_plot), figsize=(16, 14), sharex="col")

for row_idx, city in enumerate(villes):
    city_df = df[df["ville"] == city]
    for col_idx, col_name in enumerate(features_to_plot):
        ax = axes[row_idx, col_idx]
        
        if col_name == "precipitation":
            sns.histplot(data=city_df, x=col_name, hue="saison", palette=SEASONS_PALETTE,
                         element="step", stat="density", common_norm=False, bins=30, ax=ax, alpha=0.4)
            ax.set_xlim(-1, 25)  # Zoom pour une meilleure lisibilité
        else:
            sns.kdeplot(data=city_df, x=col_name, hue="saison", palette=SEASONS_PALETTE,
                        fill=True, common_norm=False, alpha=0.3, ax=ax)
            
        if col_idx == 0:
            ax.set_ylabel(f"{city}
Density")
        else:
            ax.set_ylabel("")
            
        if row_idx == 0:
            ax.set_title(f"Distribution de {col_name}")
            
        if row_idx != 0 or col_idx != 2:
            if ax.get_legend():
                ax.get_legend().remove()

fig.suptitle("Distributions des variables clés par ville et saison", y=0.99, fontsize=16)
plt.tight_layout()
plt.show()

## 6. Matrice de Corrélation

In [ ]:
corr_cols = [
    "temp_max", "temp_min", "temp_mean", "precipitation", "pluie", "heures_pluie",
    "vent_max", "rafales_max", "vent_direction", "rayonnement", "evapotranspiration",
    "ensoleillement_h", "humidite_max", "humidite_min", "humidite_mean", "rosee_max",
    "rosee_min", "ressenti_max", "ressenti_min", "pression_max", "pression_min",
    "pression_mean", "nuages_pct", "temp_sol", "humidite_sol"
]

# Garder uniquement les colonnes présentes
corr_cols = [c for c in corr_cols if c in df.columns]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=False, cmap="coolwarm", fmt=".2f",
            vmin=-1, vmax=1, center=0, square=True, linewidths=.5, cbar_kws={"shrink": .8}, ax=ax)

ax.set_title("Matrice de Corrélation des Caractéristiques Météorologiques (Triangle Inférieur)")
plt.tight_layout()
plt.show()

## 7. Visualisation de la Saisonnalité (Boxplots Mensuels par Ville)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10), sharex=True, sharey=True)
axes_flat = axes.flatten()

for idx, city in enumerate(villes):
    ax = axes_flat[idx]
    city_df = df[df["ville"] == city]
    sns.boxplot(data=city_df, x="mois", y="temp_mean", hue="saison",
                palette=SEASONS_PALETTE, dodge=False, ax=ax)
    ax.set_title(f"Saisonnalité de temp_mean - {city}")
    ax.set_xlabel("Mois")
    ax.set_ylabel("Température Moyenne (°C)" if idx % 2 == 0 else "")
    
    if idx != 3:
        ax.get_legend().remove()
    else:
        ax.legend(title="Saison", bbox_to_anchor=(1.05, 1), loc='upper left')
        
fig.suptitle("Variation Mensuelle et Saisonnière de la Température Moyenne par Ville", y=0.98)
plt.tight_layout()
plt.show()